In [1]:
import pathlib
import numpy
import polars
from data_index.iceberg_config import S3TablesCatalogConfig, IcebergTableConfig
from data_index.analysis.tables import IMOS_DATA_LIVE_TABLE
from data_index.analysis.datasets import (
    DATASET,
    get_dataset_objects_df,
    get_dataset_arrow_dataset,
)

In [3]:
table = IMOS_DATA_LIVE_TABLE.load()

In [5]:
df = table.scan(
    selected_fields=("bucket", "key", "size", "facility", "last_modified_date",),
    row_filter=(
        "facility == 'ANMN'"
    ),
).to_polars()

In [ ]:
# Extract year

df_ = (
    df
    .with_columns(
        polars.col("key").str.extract(r"\/(\d{4})\/").cast(polars.Int64).alias("year"),
        polars.col("key").str.extract(r"(\d{8})T000000Z").cast(polars.Int64).alias("file_year")
    )
    .filter(
        polars.col("year").is_not_null(),
        polars.col("year").eq(2026),
        polars.col("key").str.contains("/NRS/REAL_TIME/")
    )
    .sort(
        polars.col("file_year"),
    )
)

In [41]:
df_

bucket,key,size,last_modified_date,facility,year,file_year
str,str,i64,datetime[μs],str,i64,i64
"""imos-data""","""IMOS/ANMN/NRS/REAL_TIME/NRSDAR…",142724,2026-05-28 06:39:45,"""ANMN""",2026,20260101
"""imos-data""","""IMOS/ANMN/NRS/REAL_TIME/NRSDAR…",131068,2026-05-28 06:39:45,"""ANMN""",2026,20260201
"""imos-data""","""IMOS/ANMN/NRS/REAL_TIME/NRSDAR…",142724,2026-05-28 06:39:45,"""ANMN""",2026,20260301
"""imos-data""","""IMOS/ANMN/NRS/REAL_TIME/NRSDAR…",141572,2026-05-28 06:39:45,"""ANMN""",2026,20260401
"""imos-data""","""IMOS/ANMN/NRS/REAL_TIME/NRSDAR…",141140,2026-06-02 23:07:50,"""ANMN""",2026,20260501
…,…,…,…,…,…,…
"""imos-data""","""IMOS/ANMN/NRS/REAL_TIME/NRSYON…",70051,2026-05-28 06:49:55,"""ANMN""",2026,20260301
"""imos-data""","""IMOS/ANMN/NRS/REAL_TIME/NRSYON…",69667,2026-05-28 06:49:55,"""ANMN""",2026,20260401
"""imos-data""","""IMOS/ANMN/NRS/REAL_TIME/NRSYON…",70051,2026-06-02 23:08:00,"""ANMN""",2026,20260501
